# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
# openai = OpenAI()

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = "llama3.2"
openai = OpenAI(
    base_url="http://localhost:11434/v1",  # Trỏ về server local của Ollama
    api_key="ollama"                       # Key giả
)

In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'companies/website', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers/job page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'about the author',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog posts', 'url': 'https://edwarddonner.com/posts/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    print(f"Raw result from LLM: {result}")
    links = json.loads(result)
    print(f"links: {links}")
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Raw result from LLM: {
    "links": [
        {"type": "about page", "url": "https://edwarddonner.com/about-me-and-about-nebula/"},
        {"type": "blog", "url": "https://edwarddonner.com/posts/"},
        {"type": "Career/About the Founder", "url": "https://www.linkedin.com/in/eddonner/"},
        {"type": "Company info", "url": "https://nebula.io/?utm_source=ed&utm_medium=referral"}
    ]
}
links: {'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}, {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'}, {'type': 'Career/About the Founder', 'url': 'https://www.linkedin.com/in/eddonner/'}, {'type': 'Company info', 'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'Career/About the Founder',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Company info',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [56]:
select_relevant_links("https://huggingface.co")
# select_relevant_links("https://youtube.com")
# select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://huggingface.co by calling llama3.2
Raw result from LLM: {
    "links": [
        {"type": "about page", "url": "https://huggingface.co/"},
        {"type": "company page", "url": "https://huggingface.coBrand/"},
        {"type": "blog", "url": "https://blog.huggingface.co/"},
        {"type": "status", "url": "https://status.huggingface.co/"},
        {"type": "discuss forum", "url": "https://discuss.huggingface.co/"},
        {"type": "github", "url": "https://github.com/huggingface"},
        {"type": "twitter", "url": "https://twitter.com/huggingface"},
        {"type": "linkedin", "url": "https://www.linkedin.com/company/huggingface/"}
    ]
}
links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/'}, {'type': 'company page', 'url': 'https://huggingface.coBrand/'}, {'type': 'blog', 'url': 'https://blog.huggingface.co/'}, {'type': 'status', 'url': 'https://status.huggingface.co/'}, {'type': 'discuss forum', 'url': 'https://discus

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'company page', 'url': 'https://huggingface.coBrand/'},
  {'type': 'blog', 'url': 'https://blog.huggingface.co/'},
  {'type': 'status', 'url': 'https://status.huggingface.co/'},
  {'type': 'discuss forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
# print(fetch_page_and_all_relevant_links("https://huggingface.co"))
print(fetch_page_and_all_relevant_links("https://anthropic.com"))

Selecting relevant links for https://anthropic.com by calling llama3.2
Raw result from LLM: {
    "links": [
        {"type": "company page", "url": "https://www.anthropic.com/"},

        {"type": "about us / research page", "url": "https://www.anthropic.com/research"},
        
        {"type": "careers/jobs page", "url": "https://www.anthropic.com/careers"},
        
        {"type": "home page", "url": "https://www.anthropic.com/"},
        
        {"type": "engineering/engineering page", "url": "https://www.anthropic.com/engineering"},
        
        {"type": "news/news page", "url": "https://www.anthropic.com/news"},
        
        {"type": "features/features page", "url": "https://www.anthropic.com/features/claude-on-mars"},
        
        {"type": "e.g. economic indices page", "url": "https://www.anthropic.com/economic-index"},
        
        {"type": "constitution page", "url": "https://www.anthropic.com/constitution"},
        
        {"type": "transparency transpar

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("Anthropic", "https://anthropic.com")

Selecting relevant links for https://anthropic.com by calling llama3.2
Raw result from LLM: {
    "links": [
        {
            "type": "Company page",
            "url": "https://www.anthropic.com/"
        },
        {
            "type": "Research pages",
            "url": "https://www.anthropic.com/research"
        },
        {
            "type": "Economic futures page",
            "url": "https://www.anthropic.com/economic-futures"
        },
        {
            "type": "Constitution page",
            "url": "https://www.anthropic.com/constitution"
        },
        {
            "type": "Transparency page",
            "url": "https://www.anthropic.com/transparency"
        },
        {
            "type": "Responsible scaleing policy",
            "url": "https://www.anthropic.com/responsible-scaling-policy"
        },
        {
            "type": "Careers/Jobs page",
            "url": "https://www.anthropic.com/careers"
        },
        {
            "type": "Lea

"\nYou are looking at a company called: Anthropic\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome \\ Anthropic\n\nSkip to main content\nSkip to footer\nResearch\nEconomic Futures\nCommitments\nInitiatives\nClaude's Constitution\nTransparency\nResponsible Scaling Policy\nTrust center\nSecurity and compliance\nLearn\nLearn\nAnthropic Academy\nTutorials\nUse cases\nEngineering at Anthropic\nDeveloper docs\nCompany\nAbout\nCareers\nEvents\nNews\nTry Claude\nTry Claude\nTry Claude\nLearn more about Claude\nProducts\nClaude\nClaude Code\nClaude Cowork\nClaude Platform\nPricing\nContact sales\nModels\nOpus\nSonnet\nHaiku\nLog in\nClaude.ai\nClaude Console\nEN\nThis is some text inside of a div block.\nLog in to Claude\nLog in to Claude\nLog in to Claude\nDownload app\nDownload app\nDownload app\nResearch\nEconomic Futures\nCommitments\nInitiatives\

In [18]:
MODEL = "llama3.2"

def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("Anthropic", "https://anthropic.com")

Selecting relevant links for https://anthropic.com by calling llama3.2
Raw result from LLM: {
    "links": [
        {"type": "about page", "url": "https://www.anthropic.com/about"},
        {"type": "company page", "url": "https://www.anthropic.com/company"},
        {"type": "careers page", "url": "https://www.anthropic.com/careers"},
        {"type": "learn more page", "url": "https://www.anthropic.com/learn"},
        {"type": "engineering page", "url": "https://www.anthropic.com/engineering"},
        {"type": "news page", "url": "https://www.anthropic.com/news"},
        {"type": "events page", "url": "https://www.anthropic.com/events"},
        {"type": "constitution/policy page", "url": "https://www.anthropic.com/constitution"},
        {"type": "research page", "url": "https://www.anthropic.com/research"},
        {"type": "economic futures page", "url": "https://www.anthropic.com/economic-futures"},
        {"type": "transparency page", "url": "https://www.anthropic.com/trans

# Anthropic: Building Reliable and Safe AI Systems

At Anthropic, our mission is to ensure that artificial intelligence (AI) benefits humanity while mitigating its risks. We believe that AI will have a profound impact on the world, and it's crucial that we build systems that people can rely on.

## Our Philosophy

We treat AI safety as a systematic science, conducting research, applying it to our products, and feeding it back into our development process. Our goal is to build frontier AI systems that are reliable, interpretable, and steerable.

## Our Approach

* **Research-Driven**: We conduct cutting-edge research in AI safety, exploring new techniques and approaches to develop safe and reliable AI systems.
* **Frontier Development**: We deploy our research findings through a range of products and partnerships, ensuring that our technology is accessible to a broad audience.
* **Collaborative**: We work with experts from diverse disciplines to bring together the best ideas and expertise in the field of AI.

## Product Overview

Our flagship product, Claude, is an AI system designed to be helpful, honest, and harmless. With Claude, we aim to make AI more transparent, accountable, and trustworthy.

* **Claude Opus**: Our powerful model for coding, agents, and professional work.
* **Claude Code**: A platform for coding and developers.
* **Claude Cowork**: A collaborative space for remote workers.
* **Claude Console**: An interface for managing your Claude experience.

## Join Our Mission

At Anthropic, we're building a community of like-minded individuals passionate about making AI work for humanity. If you share our values and are drawn to hard problems with real stakes, we'd love to meet you.

### Explore Open Roles

We have a range of exciting roles available, from researcher to developer to expert engineer. Join us and contribute to shaping the future of AI.

## Stay Updated

Stay informed about the latest developments in AI safety, research, and technology by following our news and blog.

[Learn More](link to Anthropic's website)

Contact Us
(log in link)
email address

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [21]:
MODEL = "llama3.2"

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        # model="gpt-4.1-mini",
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("Anthropic", "https://anthropic.com")

Selecting relevant links for https://anthropic.com by calling llama3.2
Raw result from LLM: {
    "links": [
        {
            "type": "home page",
            "url": "https://www.anthropic.com/"
        },
        {
            "type": "About page",
            "url": "https://www.anthropic.com/company"
        },
        {
            "type": "Careers/Jobs page",
            "url": "https://www.anthropic.com/careers"
        },
        {
            "type": "Company page",
            "url": "https://www.anthropic.com"
        },
        {
            "type": "News and Blog",
            "url": "https://www.claude.com/blog"
        },
        {
            "type": "Learn page",
            "url": "https://www.anthropic.com/learn"
        },
        {
            "type": "Research page",
            "url": "https://www.anthropic.com/research"
        },
        {
            "type": "Economic Future page",
            "url": "https://www.anthropic.com/economic-futures"
        },


# Anthropic: Securing the Benefits and Mitigating the Risks of AI

At Anthropic, we're on a mission to harness the power of artificial intelligence (AI) while prioritizing its safety and societal impact. Our public benefit corporation model ensures that our primary goal is to secure the benefits of AI and mitigate its risks.

## Our Purpose

Anthropic's purpose is rooted in our commitment to ensuring that AI systems are developed, deployed, and used responsibly. We believe that AI has the potential to revolutionize numerous industries and improve human lives, but only if it's designed with safeguards that prevent harm and promote transparency.

## Core Principles

* **Safety**: We prioritize the safety of AI systems and their use cases to prevent accidents or unintended consequences.
* **Transparency**: Our approach emphasizes transparency in AI decision-making processes, model explanations, and data usage.
* **Responsibility**: As a public benefit corporation, we recognize our responsibility to ensure that AI benefits society as a whole.

## Initiatives

We're actively working on several initiatives to address these principles:

* **Project Glasswing**: Securing critical software for the AI era by identifying potential threats and developing robust solutions.
* **Claude Opus 4.6**: Unveiling the world's most powerful model for coding, agents, and professional work, while maintaining a focus on safety and security.

## Our Products

Our products are designed to promote transparency, accountability, and safety in AI systems:

* **Claude**: An open-source platform for building and deploying AI models, with a focus on explainability and trustworthiness.
* **Claude Code**: A framework for creating reliable, high-performance models using modular and flexible programming techniques.

## Join Our Community

At Anthropic, we believe that collaboration is key to advancing AI responsibly. Join our community of experts, researchers, and practitioners who share our passion for developing safe, transparent, and trustworthy AI systems.

### Careers & Events

Ready to contribute to a mission-driven company? Explore our current job openings: https://www.anthropicc.com/career/

Follow us on social media to stay updated on the latest news and events:

* Twitter: https://twitter.com/AnthropicAI
* LinkedIn: https://www.linkedin.com/company/anthropicc/
* GitHub: https://github.com/anthropic

Join our mission to harness AI's potential while ensuring that this transformative technology becomes a force for good.

In [23]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':
# Test the more humorous prompt if you like!

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [24]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Anthropic", "https://anthropic.com")

Selecting relevant links for https://anthropic.com by calling llama3.2
Raw result from LLM: {
    "links": [
        {"type": "company page", "url": "https://www.anthropic.com/"},
        {"type": "research page", "url": "https://www.anthropic.com/research"},
        {"type": "economic futures page", "url": "https://www.anthropic.com/economic-futures"},
        {"type": "constitution page", "url": "https://www.anthropic.com/constitution"},
        {"type": "transparency page", "url": "https://www.anthropic.com/transparency"},
        {"type": "responsible scaling policy page", "url": "https://www.anthropic.com/responsible-scaling-policy"},
        {"type": " careers page", "url": "https://www.anthropic.com/careers"},
        {"type": "events page", "url": "https://www.anthropic.com/events"},
        {"type": "news page", "url": "https://www.anthropic.com/news"},
        {"type": "glasswing product page", "url": "https://www.anthropic.com/glasswing"},
        {"type": "opus project page

# Anthropic: Securing the Future of Artificial Intelligence

As we continue on our journey towards a more intelligent and connected world, it's essential to acknowledge the potential risks and consequences of artificial intelligence. At Anthropic, we're committed to using our expertise to mitigate these risks and ensure that AI benefits humanity as a whole.

Our Mission
----------

Anthropic is a public benefit corporation dedicated to securing the benefits of AI while minimizing its risks. We believe that AI will have a profound impact on society, from improving healthcare and education to driving technological innovation and economic growth.

At the heart of our mission lies our research team, which comprises experts in various fields including alignment, economic research, interpretability, and societal impacts. These teams work tirelessly to understand the inner workings of AI models, identify potential risks, and develop strategies for ensuring that AI contributes positively to society.

Our Research Focus
-----------------

Anthropic's research is centered around several key areas:

*   **Alignment**: We investigate the challenges of aligning AI goals with human values, ensuring that future models promote beneficial outcomes.
*   **Economic Research**: Our team explores the economic implications of AI on industries and the broader economy, informing policies that maximize the benefits of AI for all stakeholders.
*   **Interpretability**: By uncovering how large language models work internally, we lay the foundations for safer, more transparent AI systems.
*   **Societal Impacts**: We examine the potential social implications of AI on various aspects of life, from education and employment to politics and governance.

Our Research Team
-----------------

At Anthropic, our research teams are comprised of experts with diverse backgrounds in computer science, philosophy, economics, and other relevant fields. These individuals work together to develop innovative solutions to the complex challenges posed by AI.

**Expertise Highlights:**

*   Alignment: Our alignment team leaders work closely with experts in AI development and ethics.
*   Economic Research: Economists and data scientists collaborate to quantify AI's economic impacts.
*   Interpretability: Researchers in machine learning, neuroscience, and cognitive psychology join forces to unravel the internal workings of large language models.

**Our Commitment**
----------------

At Anthropic, we're committed to transparency above all else. We believe that clarity is essential for addressing the challenges posed by AI safely.

**Transparency Highlights:**

*   **Claude's Constitution**: A comprehensive document outlining our values and principles for responsible AI development.
*   **Responsible Scaling Policy**: Guidelines for ensuring that our growth processes adhere to highest standards of sustainability.
*   **Trust Center**: Regularly updated information on our commitment to transparency, security, and compliance.

**Join Our Journey**
-----------------

If you're passionate about using AI for the greater good, we invite you to join our community. Our team is working tirelessly to address the challenges posed by AI while unlocking its limitless potential.

Explore our research pages:

['Research: Economic Futures''](https://anthropic.com/research-economic-futures/)
 ['Research: Alignments']( https://anthropic.com/research Alignment/)

Discover more about our projects and initiatives:

 ['Project Glasswing](https://anthropic.com/glasswing/)

Sign up for our newsletter to stay updated on the latest news from Anthropic.

[Join Our Community](https://discord.gg/anthropic)

Visit our website: [www.anthropiconomy.in](http://www.anthropiconomy.in)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>